In [2]:
import os
import pickle
import pandas as pd

# ================= CONFIG =================
PKL_ROOT_FOLDER = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\extra"        # root folder containing pkl files (with subfolders)
DATA_ROOT_FOLDER = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\automatic_emails_format_created"     # root folder containing csv/xlsx files (with subfolders)
OUTPUT_TXT_FILE = "unique_email_patterns.txt"

EMAIL_COLUMN = "Email Pattern"
INVALID_VALUES = {"search failed", "unknown pattern"}

# =========================================
def extract_patterns_from_pkl_tree(root_folder):
    patterns = set()

    for root, _, files in os.walk(root_folder):
        for file in files:
            if not file.lower().endswith(".pkl"):
                continue

            pkl_path = os.path.join(root, file)

            try:
                with open(pkl_path, "rb") as f:
                    data = pickle.load(f)

                if not isinstance(data, dict):
                    continue

                for value in data.values():
                    if not value:
                        continue

                    pattern = str(value).split("@", 1)[0].strip().lower()

                    if pattern and pattern not in INVALID_VALUES:
                        patterns.add(pattern)

            except Exception as e:
                print(f"❌ Failed PKL: {pkl_path} → {e}")

    return patterns


def extract_patterns_from_data_tree(root_folder):
    patterns = set()

    for root, _, files in os.walk(root_folder):
        for file in files:
            file_path = os.path.join(root, file)
            ext = os.path.splitext(file)[1].lower()

            try:
                if ext == ".csv":
                    df = pd.read_csv(file_path)
                elif ext in [".xlsx", ".xls"]:
                    df = pd.read_excel(file_path)
                else:
                    continue

                if EMAIL_COLUMN not in df.columns:
                    continue

                for value in df[EMAIL_COLUMN].dropna():
                    pattern = str(value).split("@", 1)[0].strip().lower()

                    if pattern and pattern not in INVALID_VALUES:
                        patterns.add(pattern)

            except Exception as e:
                print(f"❌ Failed DATA FILE: {file_path} → {e}")

    return patterns


def save_patterns_to_txt(patterns, output_file):
    patterns = sorted(patterns)

    with open(output_file, "w", encoding="utf-8") as f:
        for p in patterns:
            f.write(p + "\n")

    print(f"\n✅ Saved {len(patterns)} unique patterns to:")
    print(f"   {output_file}")


# ================= RUN =================
if __name__ == "__main__":

    print("📦 Scanning PKL folder tree...")
    pkl_patterns = extract_patterns_from_pkl_tree(PKL_ROOT_FOLDER)
    print(f"   → Found {len(pkl_patterns)} unique patterns from PKLs")

    print("\n📂 Scanning CSV / Excel folder tree...")
    data_patterns = extract_patterns_from_data_tree(DATA_ROOT_FOLDER)
    print(f"   → Found {len(data_patterns)} unique patterns from data files")

    final_patterns = pkl_patterns.union(data_patterns)

    print(f"\n🎯 TOTAL UNIQUE EMAIL PATTERNS: {len(final_patterns)}")

    save_patterns_to_txt(final_patterns, OUTPUT_TXT_FILE)


📦 Scanning PKL folder tree...
❌ Failed PKL: E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\extra\new_db.pkl → invalid load key, '\x00'.
❌ Failed PKL: E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\extra\old_db.pkl → invalid load key, '\x00'.
   → Found 36639 unique patterns from PKLs

📂 Scanning CSV / Excel folder tree...
   → Found 30 unique patterns from data files

🎯 TOTAL UNIQUE EMAIL PATTERNS: 36639

✅ Saved 36639 unique patterns to:
   unique_email_patterns.txt


In [1]:
import os
import pickle
from collections import Counter

BASE_PATH = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\extra"   # 👈 CHANGE THIS
OUTPUT_FILE = "email_pattern_counts.txt"



def extract_pattern(email_value: str) -> str:
    if not isinstance(email_value, str):
        return "Unknown Pattern"

    email_value = email_value.strip()

    if "@" not in email_value:
        return "Unknown Pattern"

    pattern = email_value.split("@", 1)[0].strip()
    return pattern if pattern else "Unknown Pattern"


def process_pkl_file(pkl_path: str):
    counter = Counter()
    total_count = 0

    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)

        if not isinstance(data, dict):
            return counter, total_count

        for _, email_value in data.items():
            pattern = extract_pattern(email_value)
            counter[pattern] += 1
            total_count += 1

    except Exception:
        counter["Error Reading File"] += 1
        total_count += 1

    return counter, total_count


def find_all_pkl_files(base_path: str):
    for root, _, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith(".pkl"):
                yield os.path.join(root, file)


def main():
    with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
        for pkl_file in find_all_pkl_files(BASE_PATH):
            counts, total = process_pkl_file(pkl_file)

            out.write(f"File address - {pkl_file}\n\n")
            out.write(f"Total entries - {total}\n\n")
            out.write("Counts:\n")

            for pattern, count in counts.most_common():
                out.write(f"{pattern} - {count}\n")

            out.write("\n" + "=" * 60 + "\n\n")

    print(f"✅ Done. Results saved to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


✅ Done. Results saved to: email_pattern_counts.txt


In [7]:
from collections import Counter
OUTPUT_FILE = "email_pattern_counts.txt"

INPUT_FILE = "email_pattern_counts.txt"   # 👈 existing file
OUTPUT_FILE = "global_email_pattern_summary.txt"


def is_valid_pattern_line(line: str) -> bool:
    """
    Valid pattern lines look like:
    FirstName - 20
    """
    return " - " in line and not line.lower().startswith(("file address", "total", "counts", "="))


def main():
    global_counter = Counter()
    total_entries = 0

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not is_valid_pattern_line(line):
                continue

            try:
                pattern, count = line.rsplit(" - ", 1)
                count = int(count)

                global_counter[pattern] += count
                total_entries += count

            except ValueError:
                continue  # skip malformed lines

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
        out.write("GLOBAL UNIQUE EMAIL PATTERN COUNTS\n\n")
        out.write(f"Total Unique Patterns: {len(global_counter)}\n")
        out.write(f"Total Entries: {total_entries}\n\n")

        for pattern, count in global_counter.most_common():
            out.write(f"{pattern} - {count}\n")

    print(f"✅ Global summary saved to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


✅ Global summary saved to: global_email_pattern_summary.txt


In [2]:
import os
import pickle

# ==============================
# CONFIG
# ==============================
BASE_FOLDER = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\extra\newpkl"
OUTPUT_PKL = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\extra\merged__new_2026_valid_email_patterns.pkl"

# ==============================
# VALID PATTERNS (WHITELIST)
# ==============================
VALID_PATTERNS = {
    "FirstName1LastName",
    "FirstName",
    "FirstName.LastName",
    "firstname1lastname",
    "FirstNameLastName1",
    "firstname",
    "FirstInitialLastName",
    "firstname.lastname",
    "FirstNameLastName",
    "FirstName_LastName",
    "FirstName1.LastName",
    "LastName",
    "LastNameFirstName1",
    "firstinitiallastname",
    "firstnamelastname1",
    "firstnamelastname",
    "FirstNameLastInitial",
    "FirstName.LastName1",
    "firstname_lastname",
    "LastName.FirstName",
    "lastname",
    "firstname1.lastname",
    "FirstName-LastName",
    "FirstInitial.LastName",
    "lastnamefirstname1",
    "LastName1FirstName",
    "LastNameFirstInitial",
    "firstnamelastinitial",
    "lastname.firstname",
    "lastname1firstname",
    "LastNameFirstName",
    "FirstName1_LastName",
    "LastName_FirstName",
    "FirstInitialLastInitial",
    "FirstName.LastInitial",
    "firstname.lastname1",
    "LastName-FirstName",
    "FirstName_LastName1",
    "LastName.FirstName1",
    "firstinitial.lastname",
    "firstname-lastname",
    "LastName.FirstInitial",
    "FirstInitial-LastName",
    "LastInitial-FirstName",
    "FirstName_LastInitial",
    "FirstInitial_LastName",
    "lastnamefirstinitial",
    "FirstInitial-LastInitial",
    "LastInitialFirstName",
    "LastInitial-FirstInitial",
    "LastInitialFirstInitial",
    "FirstName-LastInitial",
    "LastInitial_FirstInitial",
    "LastName_FirstInitial",
    "LastInitial.FirstName",
    "FirstInitial_LastInitial",
    "LastName-FirstInitial",
    "LastInitial_FirstName",
    "LastInitial.FirstInitial",
    "lastnamefirstname",
    "firstinitiallastinitial",
    "FirstInitial.LastInitial",
    "LastName1.FirstName",
    "firstname.lastinitial"
}

# ==============================
# HELPERS
# ==============================
def extract_pattern_and_domain(value):
    if not isinstance(value, str):
        return None, None

    if "@" not in value:
        return None, None

    pattern, domain = value.split("@", 1)
    return pattern.strip(), domain.strip()


def find_all_pkl_files(base):
    for root, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(".pkl"):
                yield os.path.join(root, f)

# ==============================
# MAIN LOGIC
# ==============================
def main():
    merged_data = {}
    total_entries = 0
    total_files = 0
    skipped_invalid = 0

    for pkl_file in find_all_pkl_files(BASE_FOLDER):
        total_files += 1

        try:
            with open(pkl_file, "rb") as f:
                data = pickle.load(f)
        except Exception:
            continue

        if not isinstance(data, dict):
            continue

        for company, value in data.items():
            total_entries += 1

            # already resolved
            if company in merged_data:
                continue

            pattern, domain = extract_pattern_and_domain(value)

            if not pattern or not domain:
                skipped_invalid += 1
                continue

            if pattern in VALID_PATTERNS:
                # ✅ SAVE FULL EMAIL (pattern + @ + domain)
                merged_data[company] = f"{pattern}@{domain}"
            else:
                skipped_invalid += 1

    # ==============================
    # SAVE OUTPUT
    # ==============================
    with open(OUTPUT_PKL, "wb") as f:
        pickle.dump(merged_data, f)

    # ==============================
    # STATS
    # ==============================
    print("✅ MERGE COMPLETE (DOMAIN PRESERVED)")
    print(f"Scanned PKL files      : {total_files}")
    print(f"Total entries scanned  : {total_entries}")
    print(f"Valid entries saved    : {len(merged_data)}")
    print(f"Invalid skipped        : {skipped_invalid}")
    print(f"Output PKL file        : {os.path.abspath(OUTPUT_PKL)}")


if __name__ == "__main__":
    main()


✅ MERGE COMPLETE (DOMAIN PRESERVED)
Scanned PKL files      : 18
Total entries scanned  : 2430659
Valid entries saved    : 371639
Invalid skipped        : 289010
Output PKL file        : E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\extra\merged__new_2026_valid_email_patterns.pkl


In [10]:
import pickle
from collections import Counter

with open(OUTPUT_PKL, "rb") as f:
    data = pickle.load(f)

pattern_counter = Counter()

for value in data.values():
    if not isinstance(value, str):
        continue

    # remove domain if exists
    pattern = value.split("@", 1)[0].strip()

    if pattern:
        pattern_counter[pattern] += 1

print("TOTAL UNIQUE PATTERNS:", len(pattern_counter))
print("\nUNIQUE EMAIL PATTERNS WITH COUNTS:\n")

for pattern, count in pattern_counter.most_common():
    print(f"{pattern} - {count}")


TOTAL UNIQUE PATTERNS: 54

UNIQUE EMAIL PATTERNS WITH COUNTS:

FirstName1LastName - 442875
FirstName - 328661
FirstName.LastName - 305565
FirstNameLastName1 - 42024
FirstInitialLastName - 37517
FirstNameLastName - 33172
FirstName_LastName - 13487
FirstName1.LastName - 11634
LastName - 11230
LastNameFirstName1 - 9151
FirstNameLastInitial - 3971
LastName1FirstName - 3417
LastName.FirstName - 3189
FirstName.LastName1 - 2367
FirstInitial.LastName - 1338
FirstName-LastName - 1149
LastNameFirstInitial - 673
LastNameFirstName - 592
FirstInitialLastInitial - 572
LastName_FirstName - 492
FirstName.LastInitial - 487
FirstName1_LastName - 457
LastName-FirstName - 315
FirstName_LastName1 - 260
firstname1lastname - 258
LastName.FirstName1 - 250
firstname.lastname - 219
firstname - 207
LastInitial-FirstName - 138
LastName.FirstInitial - 137
FirstInitial-LastName - 135
FirstInitial_LastName - 130
FirstName_LastInitial - 130
FirstInitial-LastInitial - 130
LastInitialFirstInitial - 123
LastInitialFirst

In [3]:
import pickle
import os

OLD_PKL = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\main_database\old_db_2026.pkl"
NEW_PKL = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\main_database\new_db_2026.pkl"   # 👈 change
OUTPUT_PKL = r"E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\main_database\old_updated.pkl"


def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pkl(data, path):
    with open(path, "wb") as f:
        pickle.dump(data, f)


def main():
    old_data = load_pkl(OLD_PKL)
    new_data = load_pkl(NEW_PKL)

    if not isinstance(old_data, dict) or not isinstance(new_data, dict):
        raise ValueError("Both PKL files must contain dictionaries")

    added = 0
    updated = 0

    for company, new_value in new_data.items():
        if company in old_data:
            # update existing company with NEW value
            if old_data[company] != new_value:
                old_data[company] = new_value
                updated += 1
        else:
            # add missing company
            old_data[company] = new_value
            added += 1

    save_pkl(old_data, OUTPUT_PKL)

    print("✅ MERGE COMPLETE")
    print(f"Old file entries     : {len(load_pkl(OLD_PKL))}")
    print(f"New file entries     : {len(new_data)}")
    print(f"Added companies      : {added}")
    print(f"Updated companies    : {updated}")
    print(f"Final total entries  : {len(old_data)}")
    print(f"Output file          : {os.path.abspath(OUTPUT_PKL)}")


if __name__ == "__main__":
    main()


✅ MERGE COMPLETE
Old file entries     : 1257887
New file entries     : 371639
Added companies      : 3630
Updated companies    : 31248
Final total entries  : 1261517
Output file          : E:\EmailTool-V2\Barzaan\email_creator_app\Email_Tool-2026\APP\files\main_database\old_updated.pkl
